In [1]:
import os
import sys
import urllib.request

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential, Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input

In [2]:
parser = (lambda x:datetime.datetime.strptime(x, '%Y.%m.%d')) 
df = pd.read_csv('sp_beaches_update.csv', parse_dates=['Date'])
df = df.sort_values(by=['Date'])
df=df.loc[~df['Enterococcus'].isnull()]
#remover a praia do Leste, da cidade de iguape, pois esta praia sumiu por erosão em 2012
#remover a Lagoa Prumirim, da cidade de Ubatuba, pois esta praia possui somente 3 medições
df = df.loc[df['Beach']!='DO LESTE'].loc[df['Beach']!='LAGOA PRUMIRIM']
df['praia'] = df[['City', 'Beach']].apply(lambda x: ' - '.join(x), axis=1)
print(f'Numero de praias: {len(df.praia.unique())}')
df = df[['Date','praia','Enterococcus']]
df.shape

Numero de praias: 174


(69016, 3)

In [3]:
for praia in df.praia.unique():
    df_aux = df.loc[df['praia']==praia]
    print(f'Praia: {praia} - Qtd Medições: {len(df_aux.Date)}')

Praia: BERTIOGA - BORACÉIA - COL. MARISTA - Qtd Medições: 419
Praia: UBATUBA - TONINHAS - Qtd Medições: 429
Praia: SANTOS - GONZAGA - Qtd Medições: 429
Praia: ILHABELA - ARMAÇÃO - Qtd Medições: 419
Praia: ILHA COMPRIDA - PRAINHA (BALSA) - Qtd Medições: 102
Praia: ILHA COMPRIDA - PONTAL - Qtd Medições: 102
Praia: ILHA ANCHIETA - PRAINHA DO LESTE - Qtd Medições: 345
Praia: SANTOS - JOSÉ MENINO (R. OLAVO BILAC) - Qtd Medições: 419
Praia: ILHA ANCHIETA - PRAIA DO SUL - Qtd Medições: 346
Praia: UBATUBA - ENSEADA - Qtd Medições: 429
Praia: ILHA ANCHIETA - PRAIA DE FORA - Qtd Medições: 350
Praia: SÃO VICENTE - MILIONÁRIOS - Qtd Medições: 429
Praia: SANTOS - JOSÉ MENINO (R. FREDERICO OZANAN) - Qtd Medições: 429
Praia: ILHABELA - PINTO - Qtd Medições: 419
Praia: UBATUBA - MARANDUBA - Qtd Medições: 429
Praia: ILHA ANCHIETA - PRAIA DO PRESIDIO - Qtd Medições: 350
Praia: UBATUBA - SANTA RITA - Qtd Medições: 429
Praia: SÃO SEBASTIÃO - PRAINHA - Qtd Medições: 419
Praia: BERTIOGA - ENSEADA - INDAIÁ -

Praia: UBATUBA - IPEROIG - Qtd Medições: 419
Praia: ITANHAÉM - SUARÃO - Qtd Medições: 429
Praia: ITANHAÉM - CAMPOS ELÍSEOS - Qtd Medições: 418
Praia: ITANHAÉM - JARDIM SÃO FERNANDO - Qtd Medições: 419
Praia: PERUÍBE - PERUÍBE (R. ICARAÍBA) - Qtd Medições: 419
Praia: ITANHAÉM - JARDIM CIBRATEL - Qtd Medições: 429
Praia: PERUÍBE - PERUÍBE (PARQUE TURÍSTICO) - Qtd Medições: 429
Praia: PERUÍBE - PERUÍBE (BALN. SÃO JOÃO BATISTA) - Qtd Medições: 429
Praia: PRAIA GRANDE - GUILHERMINA - Qtd Medições: 429
Praia: UBATUBA - RIO  ITAMAMBUCA - Qtd Medições: 429
Praia: ILHABELA - JULIÃO - Qtd Medições: 412
Praia: ILHA COMPRIDA - BALNEÁRIO ADRIANA - Qtd Medições: 94
Praia: ILHABELA - BARREIROS NORTE - Qtd Medições: 366
Praia: ILHABELA - BARREIROS SUL - Qtd Medições: 366
Praia: ILHABELA - ENGENHO D'ÁGUA - Qtd Medições: 346
Praia: ITANHAÉM - JARDIM REGINA - Qtd Medições: 330
Praia: GUARUJÁ - IPORANGA - Qtd Medições: 75
Praia: MONGAGUÁ - FLÓRIDA MIRIM - Qtd Medições: 321
Praia: ITANHAÉM - SUARÃO - AFPES

In [4]:
df_aux=df.loc[df['praia']=='BERTIOGA - BORACÉIA - COL. MARISTA']
d = {'praia': [praia]}
df_novo = pd.DataFrame(data=d) 
df_novo = df_novo.join(df_aux['Enterococcus'].T)
df_novo

,praia,Enterococcus
0,SÃO SEBASTIÃO - MARESIAS - TOTEM,8


In [5]:
df2=pd.DataFrame()
for praia in df.praia.unique():
    df_aux = df.loc[df['praia']==praia]
    d = {'praia': [praia]}
    df_novo = pd.DataFrame(data=d)
    for data in df_aux['Date']:
        df_novo.join(data)
    df2.append(df_novo)
df2

TypeError: 'Timestamp' object is not iterable

In [ ]:
test_size=20
treino = df.iloc[:-test_size].copy()
teste = df.iloc[-test_size:].copy()
print(treino.shape)
print(teste.shape)